In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from shapely.geometry import Point
import folium
from folium.plugins import MarkerCluster


In [ ]:
# ── Load analysed dataset ────────────────────────────────────────────────
df = pd.read_csv('food_prices_analysed.csv')
df['date'] = pd.to_datetime(df['date'])

# ── Market-level summary ──────────────────────────────────────────────────────
# One row per market — summarise all commodities tracked there
market_summary = (
    df.groupby(['region', 'district', 'market', 'latitude', 'longitude'])
    .agg(
        num_commodities  = ('commodity', 'nunique'),
        total_records    = ('price', 'count'),
        total_spikes     = ('is_spike', 'sum'),
        critical_count   = ('spike_severity', lambda x: (x == 'Critical').sum()),
        avg_price        = ('price', 'mean'),
        latest_date      = ('date', 'max'),
    )
    .reset_index()
)

# Spike rate per market
market_summary['spike_rate_pct'] = (
    market_summary['total_spikes'] / market_summary['total_records'] * 100
).round(2)

# Drop markets with no coordinates
market_summary = market_summary.dropna(subset=['latitude', 'longitude'])

print(f'Markets with coordinates: {len(market_summary)}')
print(f'Markets dropped (no coords): {df["market"].nunique() - len(market_summary)}')

# ── Convert to GeoDataFrame ───────────────────────────────────────────────────
# KEY CONCEPT: Point(longitude, latitude) — X first, then Y
# This matches the GIS convention: X = east-west, Y = north-south
geometry = [
    Point(lon, lat)
    for lon, lat in zip(market_summary['longitude'], market_summary['latitude'])
]

gdf_markets = gpd.GeoDataFrame(
    market_summary,
    geometry=geometry,
    crs='EPSG:4326'    # WGS84 — standard GPS coordinate system
)

print(f'\nGeoDataFrame created successfully')
print(f'CRS            : {gdf_markets.crs}')
print(f'Geometry type  : {gdf_markets.geom_type.unique()}')
print(f'Bounding box   : {gdf_markets.total_bounds}')
print(f'  [min_lon, min_lat, max_lon, max_lat]')
print(f'\nSample:')
print(gdf_markets[['district','market','num_commodities','total_spikes','geometry']].head(5))

In [ ]:
# ── Filter spike events only ──────────────────────────────────────────────────
spikes_df = df[df['spike_severity'] != 'Normal'].copy()
spikes_df  = spikes_df.dropna(subset=['latitude', 'longitude'])

print(f'Spike events with coordinates: {len(spikes_df):,}')

# ── Build geometry ────────────────────────────────────────────────────────────
spike_geometry = [
    Point(lon, lat)
    for lon, lat in zip(spikes_df['longitude'], spikes_df['latitude'])
]

# Select only the columns needed for the web map popup
# date must be string for GeoJSON — JSON has no datetime type
spike_cols = [
    'date', 'region', 'district', 'market',
    'commodity', 'price', 'pct_change', 'zscore', 'spike_severity'
]

gdf_spikes = gpd.GeoDataFrame(
    spikes_df[spike_cols].copy(),
    geometry=spike_geometry,
    crs='EPSG:4326'
)

# Convert date to string for GeoJSON compatibility
gdf_spikes['date'] = gdf_spikes['date'].astype(str)
gdf_spikes['pct_change'] = gdf_spikes['pct_change'].round(2)
gdf_spikes['zscore']     = gdf_spikes['zscore'].round(3)

# Severity breakdown
print(f'\nSpike severity in GeoDataFrame:')
print(gdf_spikes['spike_severity'].value_counts())

print(f'\nSample spike record:')
print(gdf_spikes[['district','market','commodity','price','pct_change','spike_severity','geometry']].head(3))

In [ ]:
# ── Export GeoJSON files ──────────────────────────────────────────────────────
gdf_markets.to_file('malawi_markets.geojson', driver='GeoJSON')
print('Saved: malawi_markets.geojson')
print('   → Open in QGIS: Layer > Add Layer > Add Vector Layer')

gdf_spikes.to_file('malawi_spikes.geojson', driver='GeoJSON')
print('Saved: malawi_spikes.geojson')
print('   → Your spike alert layer — red markers on web map')

# ── Build Folium preview map ───────────────────────────────────────────────────
# Folium is Python-wrapped Leaflet — exactly what your Phase 5 web map will use
# This preview lets you verify your data is spatially correct before QGIS

m = folium.Map(
    location=[-13.5, 34.3],   # Centre of Malawi
    zoom_start=7,
    tiles='CartoDB positron'
)

# ── Layer 1: All markets as clustered blue dots ───────────────────────────────
cluster = MarkerCluster(name='All Markets').add_to(m)

for _, row in gdf_markets.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,
        color='#1565c0',
        fill=True,
        fill_opacity=0.75,
        popup=folium.Popup(
            f"<b>{row['market']}</b><br>"
            f"District   : {row['district']}<br>"
            f"Region     : {row['region']}<br>"
            f"Commodities: {row['num_commodities']}<br>"
            f"Spike rate : {row['spike_rate_pct']}%<br>"
            f"<b>Critical spikes: {row['critical_count']}</b>",
            max_width=250
        ),
        tooltip=row['market']
    ).add_to(cluster)

# ── Layer 2: Critical spikes as pulsing red markers ───────────────────────────
critical_layer = folium.FeatureGroup(name='Critical Spike Alerts')

gdf_critical = gdf_spikes[gdf_spikes['spike_severity'] == 'Critical']

for _, row in gdf_critical.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=9,
        color='#C0392B',
        fill=True,
        fill_color='#C0392B',
        fill_opacity=0.85,
        popup=folium.Popup(
            f"<b style='color:red'>CRITICAL SPIKE</b><br>"
            f"Market    : {row['market']}<br>"
            f"District  : {row['district']}<br>"
            f"Commodity : <b>{row['commodity']}</b><br>"
            f"Price     : {row['price']:,.0f} MWK<br>"
            f"Jump      : <b style='color:red'>+{row['pct_change']:.1f}%</b><br>"
            f"Z-score   : {row['zscore']}<br>"
            f"Date      : {row['date']}",
            max_width=260
        ),
        tooltip=f"CRITICAL: {row['commodity']} +{row['pct_change']:.0f}%"
    ).add_to(critical_layer)

critical_layer.add_to(m)

# ── Layer 3: Severe spikes ─────────────────────────────────────────────────────
severe_layer = folium.FeatureGroup(name='Severe Spike Alerts')

gdf_severe = gdf_spikes[gdf_spikes['spike_severity'] == 'Severe']

for _, row in gdf_severe.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=6,
        color='#E67E22',
        fill=True,
        fill_color='#E67E22',
        fill_opacity=0.75,
        popup=folium.Popup(
            f"<b style='color:orange'>SEVERE SPIKE</b><br>"
            f"Market    : {row['market']}<br>"
            f"Commodity : <b>{row['commodity']}</b><br>"
            f"Price     : {row['price']:,.0f} MWK<br>"
            f"Jump      : +{row['pct_change']:.1f}%<br>"
            f"Date      : {row['date']}",
            max_width=240
        ),
        tooltip=f"SEVERE: {row['commodity']} +{row['pct_change']:.0f}%"
    ).add_to(severe_layer)

severe_layer.add_to(m)

# ── Layer control — toggle layers on/off ──────────────────────────────────────
folium.LayerControl(collapsed=False).add_to(m)

# ── Save and display ──────────────────────────────────────────────────────────
m.save('malawi_food_price_map.html')
print('Saved: malawi_food_price_map.html')
print('   → Open this file in Firefox to preview your web map')
print()
print('=== PHASE 1 TRULY COMPLETE ===')
print('''
Files produced:
  malawi_markets.geojson        → market point layer (QGIS + Leaflet)
  malawi_spikes.geojson         → spike alert layer  (QGIS + Leaflet)
  malawi_district_risk.csv      → choropleth input   (QGIS join)
  malawi_spikes_critical.csv    → 215 critical events
  food_prices_analysed.csv      → complete dataset
  malawi_food_price_map.html    → interactive preview map

Next: Phase 2 — QGIS + GADM district boundaries
  Download: gadm.org → Malawi → Shapefile → gadm41_MWI_2.shp
''')

# Display inline in Jupyter
m